# LLM Classification Finetuning — v0.1 Baseline

**Approach:** TF-IDF word n-grams (50k features) + response length features → LightGBM 5-fold multiclass.

This is a plumbing baseline to lock the submission pipeline. Target OOF ~1.04 (beats uniform 1.0986).

- No GPU required
- No internet required
- All dependencies available in Kaggle's default Python environment

In [ ]:
import json
import warnings
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings('ignore')

INPUT = Path('/kaggle/input/llm-classification-finetuning')
OUTPUT = Path('/kaggle/working')

print('Libraries loaded')

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
N_FOLDS = 5
N_ROUNDS = 300
SEED = 42

LGB_PARAMS = {
    'objective': 'multiclass',
    'num_class': 3,
    'metric': 'multi_logloss',
    'learning_rate': 0.05,
    'num_leaves': 63,
    'min_child_samples': 20,
    'feature_fraction': 0.7,
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'n_jobs': -1,
    'verbose': -1,
    'seed': SEED,
}

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
train = pd.read_csv(INPUT / 'train.csv')
test  = pd.read_csv(INPUT / 'test.csv')

print(f'Train: {train.shape}  Test: {test.shape}')
print(f'Train columns: {list(train.columns)}')

In [ ]:
# ── Labels ────────────────────────────────────────────────────────────────────
# 0 = winner_model_a, 1 = winner_model_b, 2 = winner_tie
y = train[['winner_model_a', 'winner_model_b', 'winner_tie']].values.argmax(axis=1).astype(np.int32)

counts = np.bincount(y)
print(f'Class counts — a:{counts[0]:,}  b:{counts[1]:,}  tie:{counts[2]:,}')

In [ ]:
# ── Feature engineering ───────────────────────────────────────────────────────
def make_text(df):
    return (
        df['prompt'].fillna('') + ' [SEP] '
        + df['response_a'].fillna('') + ' [SEP] '
        + df['response_b'].fillna('')
    )

def length_features(df):
    a = df['response_a'].fillna('').str.len().astype(float)
    b = df['response_b'].fillna('').str.len().astype(float)
    p = df['prompt'].fillna('').str.len().astype(float)
    return np.column_stack([
        a, b, p,
        a - b,
        a / (b + 1),
        (a == 0).astype(float),
        (b == 0).astype(float),
    ])

print('Fitting TF-IDF (50k word n-grams) …')
tfidf = TfidfVectorizer(
    analyzer='word', ngram_range=(1, 2), max_features=50_000,
    sublinear_tf=True, min_df=5,
)
X_tfidf_train = tfidf.fit_transform(make_text(train))
X_tfidf_test  = tfidf.transform(make_text(test))

X_train = hstack([X_tfidf_train, csr_matrix(length_features(train))])
X_test  = hstack([X_tfidf_test,  csr_matrix(length_features(test))])

print(f'X_train: {X_train.shape}  X_test: {X_test.shape}')

In [ ]:
# ── 5-fold LightGBM ───────────────────────────────────────────────────────────
oof_preds  = np.zeros((len(y), 3))
test_preds = np.zeros((len(test), 3))
fold_scores = []

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y)):
    print(f'\n── Fold {fold+1}/{N_FOLDS} ──')
    dtrain = lgb.Dataset(X_train[tr_idx], label=y[tr_idx])
    dval   = lgb.Dataset(X_train[val_idx], label=y[val_idx], reference=dtrain)

    model = lgb.train(
        LGB_PARAMS,
        dtrain,
        num_boost_round=N_ROUNDS,
        valid_sets=[dval],
        callbacks=[
            lgb.early_stopping(50, verbose=False),
            lgb.log_evaluation(50),
        ],
    )

    oof_preds[val_idx] = model.predict(X_train[val_idx])
    test_preds += model.predict(X_test) / N_FOLDS

    fold_ll = log_loss(y[val_idx], oof_preds[val_idx])
    fold_scores.append(fold_ll)
    print(f'  Fold {fold+1} log loss: {fold_ll:.4f}  (best iter: {model.best_iteration})')

oof_ll = log_loss(y, oof_preds)
print(f'\nOOF log loss: {oof_ll:.4f}')
print(f'Fold scores:  {[round(s,4) for s in fold_scores]}')

In [ ]:
# ── Write submission ──────────────────────────────────────────────────────────
sub = pd.DataFrame({
    'id':             test['id'],
    'winner_model_a': test_preds[:, 0],
    'winner_model_b': test_preds[:, 1],
    'winner_tie':     test_preds[:, 2],
})
sub.to_csv(OUTPUT / 'submission.csv', index=False)

print(f'submission.csv written — {len(sub):,} rows')
print(sub.head())
print(f'\nProbability sums (should be ~1.0): {test_preds.sum(axis=1)[:5].round(4)}')